In [1]:
import numpy as np
import pandas as pd

In [2]:
from boostie.model   import boostieModel
from boostie.data    import (
    make_regression_data,
    make_classification_data,
    make_count_data,
    train_test_split,
)
from boostie.metrics import (
    rmse, mae, r_squared,
    log_loss, accuracy,
    confusion_matrix, precision_recall,
)

## Read Titanic CSV

In [3]:
df_raw = pd.read_csv('./resources/titanic.csv')

In [4]:
df_raw.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
TARGET = "Survived"

features = ["SibSp", "Parch", "Fare", "Age"]

## Train Test Split

In [6]:
X = df_raw[features].values
y = df_raw[TARGET].values

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, seed=69)

## Train

In [8]:
model = boostieModel(
        n_estimators  = 100,
        max_depth     = 3,
        learning_rate = 0.1,
        reg_lambda    = 1.0,
        reg_gamma     = 0.0,
        objective     = "classification",
    )

In [9]:
model.fit(X_train, y_train, verbose=True)

  [round   10/100]  train loss: 0.197388
  [round   20/100]  train loss: 0.185629
  [round   30/100]  train loss: 0.180000
  [round   40/100]  train loss: 0.176568
  [round   50/100]  train loss: 0.173308
  [round   60/100]  train loss: 0.171672
  [round   70/100]  train loss: 0.171676
  [round   80/100]  train loss: 0.171677
  [round   90/100]  train loss: 0.171678
  [round  100/100]  train loss: 0.171678


XGBoostModel(objective='classification', n_estimators=100, max_depth=3, lr=0.1, lambda=1.0, gamma=0.0)

## Evaluate

In [11]:
probs = model.predict(X_test)        # probabilities
print(f"  Log-loss  : {log_loss(y_test, probs):.4f}")
print(f"  Accuracy  : {accuracy(y_test, probs):.4f}")

  Log-loss  : 0.5619
  Accuracy  : 0.7164


In [12]:
cm = confusion_matrix(y_test, probs)
print(f"           pred 0   pred 1")
print(f"  actual 0  {cm[0,0]:>5}    {cm[0,1]:>5}")
print(f"  actual 1  {cm[1,0]:>5}    {cm[1,1]:>5}")

           pred 0   pred 1
  actual 0    143       20
  actual 1     56       49


In [13]:
bins = np.linspace(0, 1, 11)
counts, _ = np.histogram(probs, bins=bins)
for i, c in enumerate(counts):
    lo, hi = bins[i], bins[i+1]
    bar = "▪" * (c // 2)
    print(f"  [{lo:.1f}–{hi:.1f}]  {c:>4}  {bar}")

  [0.0–0.1]    12  ▪▪▪▪▪▪
  [0.1–0.2]    26  ▪▪▪▪▪▪▪▪▪▪▪▪▪
  [0.2–0.3]    77  ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪
  [0.3–0.4]    46  ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪
  [0.4–0.5]    38  ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪
  [0.5–0.6]    28  ▪▪▪▪▪▪▪▪▪▪▪▪▪▪
  [0.6–0.7]    12  ▪▪▪▪▪▪
  [0.7–0.8]    18  ▪▪▪▪▪▪▪▪▪
  [0.8–0.9]     0  
  [0.9–1.0]    11  ▪▪▪▪▪
